# O formato do projeto: Delta Lake

## Task 2: escolher o formato em que os resultados ficam guardados

## Os formatos que você já conhece

| Formato | Como guarda |
|---|---|
| **CSV** | texto, uma linha por registro, sem tipos |
| **JSON** | texto, repete o nome de cada coluna em cada linha |
| **Parquet** | binário e colunar, com o schema (vídeo 4.4) |

O que falta no Parquet é **transação** (vídeo 5.1): uma escrita que falha no meio não tem "desfazer".

## Delta Lake: o Parquet com um log

O dado continua em **Parquet**; o Delta acrescenta a pasta `_delta_log/`, com um registro de cada escrita:

- **escrita atômica**: só vale quando é registrada no log, no fim;
- **versões**: as anteriores continuam disponíveis.

A sessão abaixo já vem pronta, com o pacote do Delta e duas configurações para o Spark entender tabelas Delta.

In [ ]:
import sys
sys.path.insert(0, "../../src")

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession, functions as F

construtor = (
    SparkSession.builder
    .appName("arena-delta")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(construtor).getOrCreate()

RAW = "../../data/raw"
OUT = "../../data/out"

spark

A célula abaixo já vem pronta: o mesmo código do vídeo 5.1 (`ler_partidas` e o histórico).

In [ ]:
# vídeo 5.1: as partidas do projeto
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas

def ler_partidas(pasta):
    return limpar_partidas(
        spark.read.option("header", "true").schema(MATCHES).csv(f"{pasta}/WorldCupMatches.csv")
    )

historico = ler_partidas(RAW)

A próxima célula também vem pronta: ela apaga a tabela Delta deste notebook, para as versões começarem do zero mesmo se
você rodar de novo.

In [ ]:
import shutil
shutil.rmtree(f"{OUT}/partidas_delta", ignore_errors=True)

## Gravando em Delta

Os `part-...parquet` estão lá; a novidade é o `_delta_log`, com o registro da **versão 0**.

## O segundo acidente, agora em Delta

Compare com o vídeo 5.1: o Delta grava a versão nova em arquivos novos e só no fim a registra no log.

## Voltando no tempo

A versão 0 continua inteira: é a rede de segurança que o Parquet puro não tem.

## A decisão do projeto

| Formato | Use para |
|---|---|
| **CSV** | trocar arquivos com pessoas e sistemas antigos |
| **JSON** | dado semiestruturado, respostas de APIs |
| **Parquet** | dado analítico gravado de uma vez |
| **Delta Lake** | tabelas de pipeline, regravadas, com transação e histórico |

**Decisão:** o pipeline grava em **Delta Lake**.

## Resultado da task 2

Formato escolhido. Próxima task: organizar a saída por Copa.

## Para consultar depois

- **`MERGE`:** quando o lote que chega é **parcial**, como as partidas do dia durante uma Copa, o Delta permite
  atualizar as linhas que já existem e inserir as novas numa operação só, com `MERGE INTO`. O Parquet puro não tem isso.
  No projeto, cada lote é uma Copa inteira, e o vídeo 5.3 mostra o caminho mais simples para esse caso.
- **`UPDATE` e `DELETE`:** alterar ou apagar linhas de uma tabela Delta, sem regravar tudo à mão.
- **`VACUUM`:** apaga os arquivos que não fazem parte de nenhuma versão recente. Sem ele, as versões antigas ocupam
  espaço para sempre.
- **Outros formatos de tabela:** Apache Iceberg e Apache Hudi seguem a mesma ideia do Delta, com um log ou um catálogo
  de metadados sobre arquivos Parquet.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()